### Objective: Connecting with container `db`

In [1]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType,
    ShortType, TimestampType
)
from pyspark.sql.functions import length, col, max
import os

In [2]:
spark = SparkSession.builder \
    .appName("AWS_Crawl_App") \
    .config(
        "spark.jars",
        "/home/jovyan/work/jars/postgresql-42.7.3.jar") \
    .getOrCreate()

jdbc_url = "jdbc:postgresql://postgres_db:5432/mydb"

In [3]:
spark

### Define connection properties to `postgres`
- Check connection to postgres db via `.env` setup
- EG:`# Database connection properties
db_url = "jdbc:postgresql://localhost:5432/your_database_name"`

In [4]:
## Check config of postgres
POSTGRES_HOST = "postgres" # Docker service/container name from docker compose (NOT localhost)
POSTGRES_PORT = "5432" # Internal Docker port
POSTGRES_DB   = "aws_common" # Database name from PGADMIN
#POSTGRES_DB   = "postgres" # Database name from PGADMIN

# ── Assemble the URL ─────────────────────────────────────
jdbc_url = f"jdbc:postgresql://{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"

connection_properties = {
    "user": "airflow",
    "password": "airflow",
    "driver": "org.postgresql.Driver"
}

## Read table from database

In [5]:
## Read data from postgres db
## spark.table("your_db.your_table").printSchema()

TABLE_NAME= '"DBO"."STG_AWS_CC_RAW"'
POSTGRES_USER= 'airflow'
POSTGRES_PASS= 'airflow'

## Read database table via API
# df = spark.read \
#     .format("jdbc") \
#     .option("url", jdbc_url) \
#     .option("dbtable",  '"DBO"."STG_AWS_CC_RAW"') \
#     .option("user",     POSTGRES_USER) \
#     .option("password", POSTGRES_PASS) \
#     .option("driver",   "org.postgresql.Driver") \
#     .load()

# Read database records with connection properties
df = spark.read.jdbc(
    url= jdbc_url,
    table= TABLE_NAME,
    properties= connection_properties
)

df.show(10, truncate=False)

+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------+----------------------------------------+--------------------------------------------+----------+------------+----------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+-------------------+-------------------+------------------+------------------+-----------+
|ingest_id|crawl_id                                                                        

In [6]:
df.printSchema()

root
 |-- ingest_id: string (nullable = true)
 |-- crawl_id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- url_surtkey: string (nullable = true)
 |-- url_host_registered_domain: string (nullable = true)
 |-- url_host_tld: string (nullable = true)
 |-- fetch_date: timestamp (nullable = true)
 |-- fetch_status: string (nullable = true)
 |-- content_mime_detected: string (nullable = true)
 |-- content_languages: string (nullable = true)
 |-- content_charset: string (nullable = true)
 |-- warc_filename: string (nullable = true)
 |-- warc_record_offset: string (nullable = true)
 |-- warc_record_length: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



### Create Table via Spark

In [30]:
# -- 1. create a staging table for AWS common crawl records
# -- CREATE TABLE IF NOT EXISTS "DBO"."STG_AWS_CC_RAW"
# -- (
# -- 	ingest_id varchar(255), -- primary key,
# -- 	crawl_id varchar(255), -- not null, -- EG: CC-MAIN-2024-10
# -- 	url TEXT, -- NOT NULL,
# -- 	url_surtkey TEXT,	-- canonical SURT form
# -- 	url_host_registered_domain varchar(255),
# -- 	url_host_tld VARCHAR(100),
# -- 	fetch_date TIMESTAMPTZ,
# -- 	fetch_status VARCHAR(100), -- HTTP status code
# -- 	content_mime_detected VARCHAR(255),
# -- 	content_languages VARCHAR(255),
# -- 	content_charset VARCHAR(255),
# -- 	warc_filename TEXT,
# -- 	warc_record_offset BIGINT,
# -- 	warc_record_length INT,
# -- 	ingested_at TIMESTAMPTZ DEFAULT NOW()
# -- )
# -- ;

# DROP TABLE "DBO"."STG_AWS_CC_RAW";

# CREATE TABLE IF NOT EXISTS "DBO"."STG_AWS_CC_RAW" (
#     ingest_id                   VARCHAR(1000),     --PRIMARY KEY,
#     crawl_id                    VARCHAR(5000),
#     url                         VARCHAR(20000),
#     url_surtkey                 VARCHAR(5000),
#     url_host_registered_domain  VARCHAR(5000),
#     url_host_tld                VARCHAR(5000),
#     fetch_date                  TIMESTAMPTZ,
#     fetch_status                VARCHAR(5000),
#     content_mime_detected       VARCHAR(5000),
#     content_languages           VARCHAR(5000),
#     content_charset             VARCHAR(5000),
#     warc_filename               VARCHAR(5000),
#     warc_record_offset          VARCHAR(5000),
#     warc_record_length          VARCHAR(5000),
#     ingested_at                 TIMESTAMPTZ     DEFAULT NOW()
# );

# -- NOTE: 03/24/2026: Update data length to match with AWS common crawl data max lenght

# ALTER TABLE IF EXISTS "DBO"."STG_AWS_CC_RAW"
#     OWNER to airflow;

# COMMENT ON TABLE "DBO"."STG_AWS_CC_RAW"
#     IS 'Raw staging table to store AWS common crawl records. Includes information such as mirrors WARC/CDX parsed output.';


# -- 2. Transformation- Clean and enrich records

# SELECT *
# FROM "DBO"."STG_AWS_CC_RAW";

### Read `repartition` data CSV format with enforcing `schema`

In [11]:
csv_files_path= 'data/repartition_data/'
csv_files= os.listdir(csv_files_path)
csv_files[:4]

['part-00008-ebc2286e-1137-489d-ac88-53885749dac3-c000.csv',
 'part-00004-ebc2286e-1137-489d-ac88-53885749dac3-c000.csv',
 '.part-00000-ebc2286e-1137-489d-ac88-53885749dac3-c000.csv.crc',
 '.part-00001-ebc2286e-1137-489d-ac88-53885749dac3-c000.csv.crc']

### Schema Inference on AWS common crawl data

In [12]:
## Schema configuration on columns
aws_crawl_schema = StructType([
    StructField("ingest_id",                    StringType(),       False),
    StructField("crawl_id",                     StringType(),       False),
    StructField("url",                          StringType(),       False),
    StructField("url_surtkey",                  StringType(),       True),
    StructField("url_host_registered_domain",   StringType(),       True),
    StructField("url_host_tld",                 StringType(),       True),
    StructField("fetch_date",                   TimestampType(),    True),
    StructField("fetch_status",                 ShortType(),        True),
    StructField("content_mime_detected",        StringType(),       True),
    StructField("content_languages",            StringType(),       True),
    StructField("content_charset",              StringType(),       True),
    StructField("warc_filename",                StringType(),       True),
    StructField("warc_record_offset",           StringType(),       True),
    StructField("warc_record_length",           StringType(),       True),
    StructField("ingested_at",                  TimestampType(),    True),
])

### Read `CSV` file as sample from LIST

In [13]:
df_new= spark.read\
    .format("csv")\
    .schema(aws_crawl_schema)\
    .option("header", "false")\
    .load(csv_files_path + csv_files[0])


In [15]:
df_new.show(2)

+---------+----------+--------------------+-------------------+--------------------------+---------------+----------+------------+---------------------+--------------------+------------------+-------------------+------------------+------------------+-----------+
|ingest_id|  crawl_id|                 url|        url_surtkey|url_host_registered_domain|   url_host_tld|fetch_date|fetch_status|content_mime_detected|   content_languages|   content_charset|      warc_filename|warc_record_offset|warc_record_length|ingested_at|
+---------+----------+--------------------+-------------------+--------------------------+---------------+----------+------------+---------------------+--------------------+------------------+-------------------+------------------+------------------+-----------+
|       ca|derivative|wiki)/audio_file_...|"mime": "text/html"|      "mime-detected": ...|"status": "200"|      NULL|        NULL| "offset": "628841...|"filename": "craw...|"charset": "UTF-8"|"languages": "eng"}

In [16]:
# Assuming 'df' is your PySpark DataFrame

# Create a list of aggregation expressions for all columns
# For each column 'c', calculate max(length(c)) and alias it
max_length_expressions = [max(length(col(c))).alias(f"{c}") for c in df_new.columns]

# Apply the expressions to the DataFrame using select
max_lengths_df = df_new.select(*max_length_expressions)

# Show the result
max_lengths_df.show()

+---------+--------+-----+-----------+--------------------------+------------+----------+------------+---------------------+-----------------+---------------+-------------+------------------+------------------+-----------+
|ingest_id|crawl_id|  url|url_surtkey|url_host_registered_domain|url_host_tld|fetch_date|fetch_status|content_mime_detected|content_languages|content_charset|warc_filename|warc_record_offset|warc_record_length|ingested_at|
+---------+--------+-----+-----------+--------------------------+------------+----------+------------+---------------------+-----------------+---------------+-------------+------------------+------------------+-----------+
|        2|    2454|10141|       2112|                      3019|        2119|        22|           5|                 1423|             1447|           2317|         1457|              NULL|              NULL|       NULL|
+---------+--------+-----+-----------+--------------------------+------------+----------+------------+------

In [37]:
df_new.write \
    .format("jdbc") \
    .mode("append") \
    .option("url", jdbc_url) \
    .option("dbtable", TABLE_NAME) \
    .options(**connection_properties)\
    .save()

In [ ]:
df_new.columns

### Test: Create a Table

In [ ]:
data = [Row(id=1, name="Alice"), Row(id=2, name="Bob")]
df = spark.createDataFrame(data)

df.write.jdbc(
    url=jdbc_url,
    table="people",
    mode="overwrite",       # or "append"
    properties=connection_properties
)

In [ ]:
df.show()

In [ ]:
## GET available tables in target DATABASE
db_name= 'AWS_CRAWL_URLS.DBO'
tables = spark.catalog.listTables(dbName=db_name)

### Write all CSV files into the database